In [ ]:
# Mixed-noise supervised neural decoder for a distance-3 rotated surface code
# QEC ML Experiments

In [1]:
# ============================================================
# Experiment: Mixed-Noise Neural Decoder
# ============================================================

import os
import random
import numpy as np
import torch

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE_DIR = os.path.expanduser("~/Desktop/qskit/qec_ml")

TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")
TEST_DIR = os.path.join(BASE_DIR, "data", "test")

MODEL_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


# ------------------------------------------------------------
# Experiment parameters
# ------------------------------------------------------------

DISTANCE = 3

NOISE_RATES = [0.005, 0.010, 0.020]

TRAIN_SHOTS_PER_RATE = 50_000
TEST_SHOTS_PER_RATE = 100_000

VALIDATION_FRACTION = 0.10


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print("=" * 70)
print("EXPERIMENT CONFIGURATION")
print("=" * 70)

print(f"Base directory:       {BASE_DIR}")
print(f"Code distance:        {DISTANCE}")
print(f"Noise rates:          {NOISE_RATES}")
print(f"Train samples/rate:   {TRAIN_SHOTS_PER_RATE:,}")
print(f"Total train samples:  {TRAIN_SHOTS_PER_RATE * len(NOISE_RATES):,}")
print(f"Test samples/rate:    {TEST_SHOTS_PER_RATE:,}")
print(f"Validation fraction:  {VALIDATION_FRACTION}")
print(f"Random seed:          {SEED}")
print(f"Device:               {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU:                  {torch.cuda.get_device_name(0)}")

print("=" * 70)

EXPERIMENT CONFIGURATION
Base directory:       /home/ankit/Desktop/qskit/qec_ml
Code distance:        3
Noise rates:          [0.005, 0.01, 0.02]
Train samples/rate:   50,000
Total train samples:  150,000
Test samples/rate:    100,000
Validation fraction:  0.1
Random seed:          42
Device:               cuda
GPU:                  NVIDIA GeForce RTX 3050 A Laptop GPU


In [2]:
# ============================================================
# Loading validated datasets
# ============================================================

def load_dataset(directory, noise_rate):
    filename = f"d3_p{int(noise_rate * 1000):03d}.npz"
    path = os.path.join(directory, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found: {path}")

    data = np.load(path)

    X = data["X"]
    y = data["y"]

    metadata = {
        key: data[key].item()
        for key in data.files
        if key not in ["X", "y"]
    }

    return X, y, metadata


train_data = {}
test_data = {}

for p in NOISE_RATES:
    X_train, y_train, meta_train = load_dataset(TRAIN_DIR, p)
    X_test, y_test, meta_test = load_dataset(TEST_DIR, p)

    train_data[p] = {
        "X": X_train,
        "y": y_train,
        "metadata": meta_train
    }

    test_data[p] = {
        "X": X_test,
        "y": y_test,
        "metadata": meta_test
    }


print("=" * 70)
print("DATASETS LOADED")
print("=" * 70)

for p in NOISE_RATES:
    print(
        f"p={p:.3f} | "
        f"train={train_data[p]['X'].shape} | "
        f"test={test_data[p]['X'].shape}"
    )

print("=" * 70)

DATASETS LOADED
p=0.005 | train=(50000, 24) | test=(100000, 24)
p=0.010 | train=(50000, 24) | test=(100000, 24)
p=0.020 | train=(50000, 24) | test=(100000, 24)


In [ ]:
# Our model sees only:

# $$ \mathbf{x} = (x_1,x_2,\ldots,x_{24}) $$

# where the \(x_i\)'s are detector outcomes.

# It does not get:

# $$ p $$

# as an additional feature.

# That makes the experiment more interesting:

# Can the decoder infer the logical correction from the syndrome alone across multiple physical noise regimes?

In [3]:
# ============================================================
# Combine training datasets
# ============================================================

X_train_all = np.concatenate(
    [train_data[p]["X"] for p in NOISE_RATES],
    axis=0
)

y_train_all = np.concatenate(
    [train_data[p]["y"] for p in NOISE_RATES],
    axis=0
)

# Track the physical noise rate for analysis
p_train_all = np.concatenate([
    np.full(len(train_data[p]["y"]), p)
    for p in NOISE_RATES
])


print("=" * 70)
print("COMBINED TRAINING DATA")
print("=" * 70)

print(f"X shape:              {X_train_all.shape}")
print(f"y shape:              {y_train_all.shape}")
print(f"Total samples:        {len(y_train_all):,}")
print(f"Number of detectors:  {X_train_all.shape[1]}")

print()
print("Overall class distribution:")
print(f"Class 0:              {(y_train_all == 0).sum():,}")
print(f"Class 1:              {(y_train_all == 1).sum():,}")
print(f"Positive rate:        {y_train_all.mean():.6f}")

print()
print("Samples by noise rate:")

for p in NOISE_RATES:
    mask = p_train_all == p

    print(
        f"p={p:.3f} | "
        f"samples={mask.sum():,} | "
        f"positive rate={y_train_all[mask].mean():.6f}"
    )

print("=" * 70)

COMBINED TRAINING DATA
X shape:              (150000, 24)
y shape:              (150000,)
Total samples:        150,000
Number of detectors:  24

Overall class distribution:
Class 0:              120,284
Class 1:              29,716
Positive rate:        0.198107

Samples by noise rate:
p=0.005 | samples=50,000 | positive rate=0.105940
p=0.010 | samples=50,000 | positive rate=0.186140
p=0.020 | samples=50,000 | positive rate=0.302240


In [ ]:
# We'll split only the 150k training data. The three 100k test sets remain completely untouched.

# I also want the split to preserve the noise-rate composition, so we'll stratify using the combination of:

# noise rate \(p\)
# logical label \(y\)

# This prevents the validation set from accidentally having a different mixture of noise levels.

In [4]:
# ============================================================
# Train / Validation Split
# ============================================================

from sklearn.model_selection import train_test_split

# Create a stratification label combining:
#   - physical noise rate
#   - logical target
#
# This preserves both the noise-rate distribution and
# class distribution in train and validation sets.

stratify_labels = np.array([
    f"{p:.3f}_{label}"
    for p, label in zip(p_train_all, y_train_all)
])

X_train, X_val, y_train, y_val, p_train, p_val = train_test_split(
    X_train_all,
    y_train_all,
    p_train_all,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=stratify_labels
)


print("=" * 70)
print("TRAIN / VALIDATION SPLIT")
print("=" * 70)

print(f"Training X shape:       {X_train.shape}")
print(f"Training y shape:       {y_train.shape}")
print(f"Validation X shape:     {X_val.shape}")
print(f"Validation y shape:     {y_val.shape}")

print()
print("Overall class distribution:")

print(
    f"Train      | "
    f"class 0={np.sum(y_train == 0):,} | "
    f"class 1={np.sum(y_train == 1):,} | "
    f"positive={y_train.mean():.6f}"
)

print(
    f"Validation | "
    f"class 0={np.sum(y_val == 0):,} | "
    f"class 1={np.sum(y_val == 1):,} | "
    f"positive={y_val.mean():.6f}"
)

print()
print("Distribution by noise rate:")

for p in NOISE_RATES:

    train_mask = p_train == p
    val_mask = p_val == p

    print(
        f"p={p:.3f} | "
        f"train={train_mask.sum():,} "
        f"(positive={y_train[train_mask].mean():.6f}) | "
        f"validation={val_mask.sum():,} "
        f"(positive={y_val[val_mask].mean():.6f})"
    )

print("=" * 70)

TRAIN / VALIDATION SPLIT
Training X shape:       (135000, 24)
Training y shape:       (135000,)
Validation X shape:     (15000, 24)
Validation y shape:     (15000,)

Overall class distribution:
Train      | class 0=108,256 | class 1=26,744 | positive=0.198104
Validation | class 0=12,028 | class 1=2,972 | positive=0.198133

Distribution by noise rate:
p=0.005 | train=45,000 (positive=0.105933) | validation=5,000 (positive=0.106000)
p=0.010 | train=45,000 (positive=0.186133) | validation=5,000 (positive=0.186200)
p=0.020 | train=45,000 (positive=0.302244) | validation=5,000 (positive=0.302200)


In [ ]:
    # current structure
       #           150,000 TRAINING
       #                 │
       #          ┌──────┴──────┐
       #          │             │
       #       135,000       15,000
       #        TRAIN          VAL
       #          │             │
       #          └──────┬──────┘
       #                 │
       #           model selection
       #                 │
       #                 ▼
       #           FINAL MODEL
       #                 │
       #    ┌────────────┼────────────┐
       #    ▼            ▼            ▼
       # p=.005       p=.010       p=.020
       # 100k TEST    100k TEST    100k TEST


# Establish the majority-class baseline
# we need a trivial baseline.
# Since class 0 is the majority: y^= 0∀x.

In [5]:
# ============================================================
# Majority-Class Baseline
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Always predict the majority class: 0
majority_prediction = np.zeros_like(y_val)

baseline_accuracy = accuracy_score(y_val, majority_prediction)
baseline_balanced_accuracy = balanced_accuracy_score(
    y_val, majority_prediction
)
baseline_precision = precision_score(
    y_val, majority_prediction,
    zero_division=0
)
baseline_recall = recall_score(
    y_val, majority_prediction,
    zero_division=0
)
baseline_f1 = f1_score(
    y_val, majority_prediction,
    zero_division=0
)
baseline_cm = confusion_matrix(
    y_val, majority_prediction
)


print("=" * 70)
print("MAJORITY-CLASS BASELINE")
print("=" * 70)

print(f"Accuracy:           {baseline_accuracy:.6f}")
print(f"Balanced accuracy:  {baseline_balanced_accuracy:.6f}")
print(f"Precision:          {baseline_precision:.6f}")
print(f"Recall:             {baseline_recall:.6f}")
print(f"F1:                 {baseline_f1:.6f}")

print()
print("Confusion matrix:")
print(baseline_cm)

print("=" * 70)

MAJORITY-CLASS BASELINE
Accuracy:           0.801867
Balanced accuracy:  0.500000
Precision:          0.000000
Recall:             0.000000
F1:                 0.000000

Confusion matrix:
[[12028     0]
 [ 2972     0]]


In [ ]:
# The exact accuracy follows directly from the validation distribution:

# $$ \frac{12028}{15000}=0.801867. $$
# Why this matters

# Our neural decoder must beat:

# $$ \boxed{\text{Accuracy}=80.19\%} $$
# but more importantly it must beat:
# $$ \boxed{\text{Balanced Accuracy}=50\%} $$  by a substantial margin.
# A model that gets 90% accuracy simply by becoming conservative about predicting logical errors isn't particularly useful.

# Calculate the training class weight

# Our training data has:

# Class 0 = 108,256
# Class 1 = 26,744

# So class 1 is substantially underrepresented.

# We'll use a standard balanced weighting: $$ w_c = \frac{N}{2N_c}. $

# Therefore:
# $$ w_0=\frac{135000}{2(108256)} $$ and $$w_1=\frac{135000}{2(26744)}$$.

In [6]:
# ============================================================
# Class Weights
# ============================================================

class_0_count = np.sum(y_train == 0)
class_1_count = np.sum(y_train == 1)

n_train = len(y_train)

weight_0 = n_train / (2.0 * class_0_count)
weight_1 = n_train / (2.0 * class_1_count)

class_weights = torch.tensor(
    [weight_0, weight_1],
    dtype=torch.float32,
    device=DEVICE
)

positive_class_weight = weight_1


print("=" * 70)
print("CLASS WEIGHTS")
print("=" * 70)

print(f"Class 0 count:       {class_0_count:,}")
print(f"Class 1 count:       {class_1_count:,}")

print()
print(f"Weight for class 0:  {weight_0:.6f}")
print(f"Weight for class 1:  {weight_1:.6f}")

print()
print(f"Positive class weight: {positive_class_weight:.6f}")

print("=" * 70)

CLASS WEIGHTS
Class 0 count:       108,256
Class 1 count:       26,744

Weight for class 0:  0.623522
Weight for class 1:  2.523931

Positive class weight: 2.523931


In [ ]:
# This model will be trained on all three noise regimes simultaneously, and we'll select the best checkpoint using validation balanced accuracy.

In [7]:
# ============================================================
# Convert datasets to PyTorch tensors
# ============================================================

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32
)


print("=" * 70)
print("PYTORCH DATA")
print("=" * 70)

print(f"X_train: {X_train_tensor.shape}")
print(f"y_train: {y_train_tensor.shape}")
print(f"X_val:   {X_val_tensor.shape}")
print(f"y_val:   {y_val_tensor.shape}")

print()
print(f"X_train dtype: {X_train_tensor.dtype}")
print(f"y_train dtype: {y_train_tensor.dtype}")

print("=" * 70)

PYTORCH DATA
X_train: torch.Size([135000, 24])
y_train: torch.Size([135000])
X_val:   torch.Size([15000, 24])
y_val:   torch.Size([15000])

X_train dtype: torch.float32
y_train dtype: torch.float32


In [8]:
# ============================================================
# Create DataLoaders
# We'll use mini-batch training rather than feeding all 135,000 samples through the network at once.
# PyTorch DataLoaders
# ============================================================

from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 512

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("=" * 70)
print("DATALOADERS")
print("=" * 70)

print(f"Batch size:          {BATCH_SIZE}")
print(f"Training batches:    {len(train_loader)}")
print(f"Validation batches:  {len(val_loader)}")
print(f"Shuffle training:    True")
print(f"Shuffle validation:  False")

print("=" * 70)

DATALOADERS
Batch size:          512
Training batches:    264
Validation batches:  30
Shuffle training:    True
Shuffle validation:  False


In [9]:
# MLP V2
# 24 detector events
#        ↓
#      Linear
#        ↓
#       ReLU
#        ↓
#      64
#        ↓
#      Linear
#        ↓
#       ReLU
#        ↓
#      32
#        ↓
#      Linear
#        ↓
#       1 logit

In [10]:
# ============================================================
# MLP v2 — Neural QEC Decoder
# ============================================================

import torch.nn as nn


class QECMLP(nn.Module):

    def __init__(self, input_dim=24):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


model = QECMLP(
    input_dim=X_train.shape[1]
).to(DEVICE)


print("=" * 70)
print("MLP v2")
print("=" * 70)

print(model)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print()
print(f"Total parameters:      {total_parameters:,}")
print(f"Trainable parameters:  {trainable_parameters:,}")
print(f"Device:                {DEVICE}")

print("=" * 70)

MLP v2
QECMLP(
  (network): Sequential(
    (0): Linear(in_features=24, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total parameters:      3,713
Trainable parameters:  3,713
Device:                cuda


In [ ]:
# Weighted loss and optimizer

# We'll use BCEWithLogitsLoss.

# Rather than manually applying class weights to every sample, we can use the positive-class weight:

# $$ \text{pos\_weight}=\frac{N_0}{N_1}. $$

# This gives approximately:

# $$ \frac{108256}{26744}\approx4.047. $$

# However, notice something important:

# Previous Cell 6 calculated balanced class weights \(w_0,w_1\). Those weights should not be passed directly as pos_weight.

# pos_weight specifically means:

# $$ \frac{\text{weight of positive examples}} {\text{weight of negative examples}}. $$

# Therefore:
# pos_weight=w_0/w_1=2.523931/0.623522≈4.047.

In [11]:
# ============================================================
# Loss Function and Optimizer
# ============================================================

pos_weight_value = weight_1 / weight_0

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(f"Positive class weight: {pos_weight_value:.6f}")
print(f"Learning rate:         {LEARNING_RATE}")
print(f"Optimizer:             Adam")
print(f"Loss:                  BCEWithLogitsLoss")
print("=" * 70)

TRAINING CONFIGURATION
Positive class weight: 4.047861
Learning rate:         0.001
Optimizer:             Adam
Loss:                  BCEWithLogitsLoss


In [ ]:
# The earlier class-weight calculation was useful for understanding the imbalance, but for PyTorch's BCEWithLogitsLoss(pos_weight=...), we want the ratio:
# $$ \boxed{ \text{pos\_weight} = \frac{N_0}{N_1} \approx4.047 } $$ and not 2.523931.


# Our model-selection criterion is explicitly:

# $$ \boxed{\text{Validation Balanced Accuracy}} $$

# because the dataset is imbalanced.

# Why checkpointing matters

# Suppose we get:

# Epoch 17 → 0.887
# Epoch 18 → 0.891
# Epoch 19 → 0.894  <-- best
# Epoch 20 → 0.892
# Epoch 21 → 0.889
# ...

# The final model should not be epoch 21.

# It should be the saved epoch-19 model.

# That's what mlp_v2_best.pt gives us.

In [16]:
# ============================================================
# Training Loop
# ============================================================

from sklearn.metrics import balanced_accuracy_score


MAX_EPOCHS = 50
PATIENCE = 8

best_val_balanced_accuracy = -np.inf
best_epoch = 0
epochs_without_improvement = 0

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_balanced_accuracy": []
}

checkpoint_path = os.path.join(
    MODEL_DIR,
    "mlp_v2_best.pt"
)


print("=" * 70)
print("STARTING MLP v2 TRAINING")
print("=" * 70)

for epoch in range(1, MAX_EPOCHS + 1):

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()

    running_train_loss = 0.0
    train_samples = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        optimizer.step()

        batch_size = X_batch.size(0)

        running_train_loss += loss.item() * batch_size
        train_samples += batch_size

    train_loss = running_train_loss / train_samples


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    model.eval()

    running_val_loss = 0.0
    val_samples = 0

    all_val_predictions = []
    all_val_targets = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            logits = model(X_batch)

            loss = criterion(logits, y_batch)

            probabilities = torch.sigmoid(logits)

            predictions = (
                probabilities >= 0.5
            ).long()

            batch_size = X_batch.size(0)

            running_val_loss += loss.item() * batch_size
            val_samples += batch_size

            all_val_predictions.append(
                predictions.cpu().numpy()
            )

            all_val_targets.append(
                y_batch.cpu().numpy().astype(int)
            )


    val_loss = running_val_loss / val_samples

    all_val_predictions = np.concatenate(
        all_val_predictions
    )

    all_val_targets = np.concatenate(
        all_val_targets
    )

    val_balanced_accuracy = balanced_accuracy_score(
        all_val_targets,
        all_val_predictions
    )


    # --------------------------------------------------------
    # Store history
    # --------------------------------------------------------

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_balanced_accuracy"].append(
        val_balanced_accuracy
    )


    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    improved = (
        val_balanced_accuracy
        > best_val_balanced_accuracy
    )

    if improved:

        best_val_balanced_accuracy = (
            val_balanced_accuracy
        )

        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "val_balanced_accuracy":
                    val_balanced_accuracy,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "seed": SEED,
                "distance": DISTANCE,
                "noise_rates": NOISE_RATES,
                "input_dim": X_train.shape[1],
            },
            checkpoint_path
        )

        marker = "  <-- BEST"

    else:

        epochs_without_improvement += 1

        marker = ""


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Val Balanced Acc: "
        f"{val_balanced_accuracy:.6f}"
        f"{marker}"
    )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print()
        print(
            f"Early stopping triggered after "
            f"{PATIENCE} epochs without improvement."
        )

        break


print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(f"Best epoch:                 {best_epoch}")
print(
    f"Best validation balanced accuracy: "
    f"{best_val_balanced_accuracy:.6f}"
)
print(f"Checkpoint:                 {checkpoint_path}")
print("=" * 70)

STARTING MLP v2 TRAINING
Epoch 01/50 | Train Loss: 0.866759 | Val Loss: 0.661671 | Val Balanced Acc: 0.817862  <-- BEST
Epoch 02/50 | Train Loss: 0.616139 | Val Loss: 0.571652 | Val Balanced Acc: 0.843854  <-- BEST
Epoch 03/50 | Train Loss: 0.560213 | Val Loss: 0.534396 | Val Balanced Acc: 0.856689  <-- BEST
Epoch 04/50 | Train Loss: 0.528619 | Val Loss: 0.513211 | Val Balanced Acc: 0.859947  <-- BEST
Epoch 05/50 | Train Loss: 0.508053 | Val Loss: 0.497558 | Val Balanced Acc: 0.862840  <-- BEST
Epoch 06/50 | Train Loss: 0.493833 | Val Loss: 0.485224 | Val Balanced Acc: 0.867335  <-- BEST
Epoch 07/50 | Train Loss: 0.483393 | Val Loss: 0.482524 | Val Balanced Acc: 0.868015  <-- BEST
Epoch 08/50 | Train Loss: 0.476199 | Val Loss: 0.475579 | Val Balanced Acc: 0.870791  <-- BEST
Epoch 09/50 | Train Loss: 0.470323 | Val Loss: 0.469898 | Val Balanced Acc: 0.872269  <-- BEST
Epoch 10/50 | Train Loss: 0.464986 | Val Loss: 0.468857 | Val Balanced Acc: 0.872908  <-- BEST
Epoch 11/50 | Train Loss:

In [ ]:
# The best model is:

# $$ \boxed{\text{Epoch 18}} $$

# with:

# $$ \boxed{\text{Validation balanced accuracy}=0.879943} $$

# Compared with the majority baseline:

# $$ 0.500000 $$

# that's a substantial improvement:

# $$ 0.879943-0.500000=\boxed{0.379943} $$

# So the MLP is clearly learning meaningful information from the 24 detector outcomes.

# The training also shows a sensible learning pattern:

# Rapid improvement during epochs 1–10.
# Continued improvement, but slower, through epoch 18.
# After epoch 18, validation balanced accuracy fluctuates around \(0.876-0.879\).
# Training loss continues decreasing while validation performance stops improving.
# Early stopping at epoch 26 is therefore reasonable.

# Importantly, we should not interpret the 0.879943 as the final performance of the decoder. It is only the validation performance used for model selection.

In [ ]:
# final evaluation on untouched test sets

# Now comes the important experiment.

# We will load:

# models/mlp_v2_best.pt

# and evaluate it independently on:

# p = 0.005 → 100,000 test shots
# p = 0.010 → 100,000 test shots
# p = 0.020 → 100,000 test shots

# We should report the metrics separately rather than mixing the test sets together. Otherwise we could hide whether the decoder works well at low or high noise.

In [17]:
# ============================================================
# Load Best MLP v2 Checkpoint
# ============================================================

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)
# We use weights_only=False here because this is the checkpoint we created ourselves and it contains the additional experiment metadata.

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("=" * 70)
print("BEST CHECKPOINT LOADED")
print("=" * 70)

print(f"Checkpoint:              {checkpoint_path}")
print(f"Best epoch:              {checkpoint['epoch']}")
print(
    f"Validation balanced acc: "
    f"{checkpoint['val_balanced_accuracy']:.6f}"
)
print(f"Model device:            {DEVICE}")

print("=" * 70)

BEST CHECKPOINT LOADED
Checkpoint:              /home/ankit/Desktop/qskit/qec_ml/models/mlp_v2_best.pt
Best epoch:              18
Validation balanced acc: 0.879943
Model device:            cuda


In [18]:
# ============================================================
# Final Evaluation on Untouched Test Sets
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def evaluate_model(model, X, y, device):
    """
    Evaluate a trained binary classifier on a dataset.
    """

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y,
        dtype=torch.float32
    )

    loader = DataLoader(
        TensorDataset(X_tensor, y_tensor),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    all_predictions = []
    all_probabilities = []
    all_targets = []

    model.eval()

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(
                device,
                non_blocking=True
            )

            logits = model(X_batch)

            probabilities = torch.sigmoid(logits)

            predictions = (
                probabilities >= 0.5
            ).long()

            all_probabilities.append(
                probabilities.cpu().numpy()
            )

            all_predictions.append(
                predictions.cpu().numpy()
            )

            all_targets.append(
                y_batch.numpy().astype(int)
            )

    probabilities = np.concatenate(
        all_probabilities
    )

    predictions = np.concatenate(
        all_predictions
    )

    targets = np.concatenate(
        all_targets
    )

    metrics = {
        "accuracy": accuracy_score(
            targets,
            predictions
        ),

        "balanced_accuracy": balanced_accuracy_score(
            targets,
            predictions
        ),

        "precision": precision_score(
            targets,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            targets,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            targets,
            predictions,
            zero_division=0
        ),

        "confusion_matrix": confusion_matrix(
            targets,
            predictions
        ),

        "positive_rate_actual": targets.mean(),

        "positive_rate_predicted": predictions.mean()
    }

    return metrics, probabilities, predictions


# ------------------------------------------------------------
# Evaluate each noise rate independently
# ------------------------------------------------------------

test_results = {}

print("=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)

for p in NOISE_RATES:

    X_test = test_data[p]["X"]
    y_test = test_data[p]["y"]

    metrics, probabilities, predictions = evaluate_model(
        model,
        X_test,
        y_test,
        DEVICE
    )

    test_results[p] = {
        "metrics": metrics,
        "probabilities": probabilities,
        "predictions": predictions
    }

    print()
    print("-" * 70)
    print(f"NOISE RATE p = {p:.3f}")
    print("-" * 70)

    print(
        f"Accuracy:             "
        f"{metrics['accuracy']:.6f}"
    )

    print(
        f"Balanced accuracy:    "
        f"{metrics['balanced_accuracy']:.6f}"
    )

    print(
        f"Precision:             "
        f"{metrics['precision']:.6f}"
    )

    print(
        f"Recall:                "
        f"{metrics['recall']:.6f}"
    )

    print(
        f"F1:                    "
        f"{metrics['f1']:.6f}"
    )

    print(
        f"Actual positive rate:  "
        f"{metrics['positive_rate_actual']:.6f}"
    )

    print(
        f"Predicted positive:    "
        f"{metrics['positive_rate_predicted']:.6f}"
    )

    print()
    print("Confusion matrix:")
    print(metrics["confusion_matrix"])

print()
print("=" * 70)
print("FINAL TEST EVALUATION COMPLETE")
print("=" * 70)

FINAL TEST EVALUATION

----------------------------------------------------------------------
NOISE RATE p = 0.005
----------------------------------------------------------------------
Accuracy:             0.965810
Balanced accuracy:    0.957954
Precision:             0.776411
Recall:                0.948004
F1:                    0.853670
Actual positive rate:  0.105200
Predicted positive:    0.128450

Confusion matrix:
[[86608  2872]
 [  547  9973]]

----------------------------------------------------------------------
NOISE RATE p = 0.010
----------------------------------------------------------------------
Accuracy:             0.891260
Balanced accuracy:    0.897362
Precision:             0.649086
Recall:                0.907092
F1:                    0.756701
Actual positive rate:  0.186420
Predicted positive:    0.260520

Confusion matrix:
[[72216  9142]
 [ 1732 16910]]

----------------------------------------------------------------------
NOISE RATE p = 0.020
-------------

In [ ]:
       #              Same Stim data
       #                    │
       #       ┌────────────┴────────────┐
       #       │                         │
       #       ▼                         ▼
       # PyMatching                 Neural MLP
       # classical                 supervised ML
       #       │                         │
       #       └────────────┬────────────┘
       #                    ▼
       #           Compare decoding
       #              performance
# Our labels \(y\) come from Stim's logical observable, while \(X\) consists of detector events.
# So conceptually our supervised-learning problem is:
# $$ \boxed{ \text{detector syndrome} \longrightarrow \text{logical observable} } $$
# PyMatching can operate directly on the detector error model and predict the logical observable.

In [19]:
# ============================================================
# PyMatching Baseline
# Check installation
# ============================================================

import stim
import pymatching

print("=" * 70)
print("PYMATCHING ENVIRONMENT")
print("=" * 70)

print(f"Stim version:        {stim.__version__}")
print(f"PyMatching version:  {pymatching.__version__}")

print("=" * 70)

PYMATCHING ENVIRONMENT
Stim version:        1.16.0
PyMatching version:  2.4.0


In [20]:
# ============================================================
# Construct Stim Detector Error Model
# ============================================================

def build_qec_circuit(distance, rounds, noise_rate):
    """
    Reconstruct the exact Stim circuit used for dataset
    generation.
    """

    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        distance=distance,
        rounds=rounds,
        after_clifford_depolarization=noise_rate,
        before_round_data_depolarization=noise_rate,
        before_measure_flip_probability=noise_rate,
        after_reset_flip_probability=noise_rate,
    )

    return circuit


def build_matching(distance, rounds, noise_rate):

    circuit = build_qec_circuit(
        distance=distance,
        rounds=rounds,
        noise_rate=noise_rate
    )

    detector_error_model = (
        circuit.detector_error_model(
            decompose_errors=True
        )
    )

    matching = pymatching.Matching.from_detector_error_model(
        detector_error_model
    )

    return circuit, detector_error_model, matching


print("=" * 70)
print("BUILDING PYMATCHING MODELS")
print("=" * 70)

matching_models = {}

for p in NOISE_RATES:

    circuit, dem, matching = build_matching(
        distance=DISTANCE,
        rounds=3,
        noise_rate=p
    )

    matching_models[p] = {
        "circuit": circuit,
        "dem": dem,
        "matching": matching
    }

    print(
        f"p={p:.3f} | "
        f"detectors={dem.num_detectors} | "
        f"observables={dem.num_observables}"
    )

print("=" * 70)

BUILDING PYMATCHING MODELS
p=0.005 | detectors=24 | observables=1
p=0.010 | detectors=24 | observables=1
p=0.020 | detectors=24 | observables=1


In [21]:
# ============================================================
# PyMatching Test Evaluation
# ============================================================

py_matching_results = {}

print("=" * 70)
print("PYMATCHING TEST EVALUATION")
print("=" * 70)

for p in NOISE_RATES:

    X_test = test_data[p]["X"]
    y_test = test_data[p]["y"]

    matching = matching_models[p]["matching"]

    # Decode all detector syndromes
    predictions = matching.decode_batch(X_test)

    # For one logical observable, convert
    # (N, 1) -> (N,)
    predictions = np.asarray(predictions).reshape(-1)

    predictions = predictions.astype(int)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    cm = confusion_matrix(
        y_test,
        predictions
    )

    py_matching_results[p] = {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
        "predictions": predictions
    }

    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print()
    print("-" * 70)
    print(f"NOISE RATE p = {p:.3f}")
    print("-" * 70)

    print(f"Accuracy:             {accuracy:.6f}")
    print(f"Balanced accuracy:    {balanced_accuracy:.6f}")
    print(f"Precision:            {precision:.6f}")
    print(f"Recall:               {recall:.6f}")
    print(f"F1:                   {f1:.6f}")

    print()
    print("Actual positive rate:   "
          f"{y_test.mean():.6f}")

    print("Predicted positive rate: "
          f"{predictions.mean():.6f}")

    print()
    print("Confusion matrix:")
    print(cm)

print()
print("=" * 70)
print("PYMATCHING EVALUATION COMPLETE")
print("=" * 70)

PYMATCHING TEST EVALUATION

----------------------------------------------------------------------
NOISE RATE p = 0.005
----------------------------------------------------------------------
Accuracy:             0.981130
Balanced accuracy:    0.946131
Precision:            0.917416
Recall:               0.901806
F1:                   0.909544

Actual positive rate:   0.105200
Predicted positive rate: 0.103410

Confusion matrix:
[[88626   854]
 [ 1033  9487]]

----------------------------------------------------------------------
NOISE RATE p = 0.010
----------------------------------------------------------------------
Accuracy:             0.936050
Balanced accuracy:    0.889244
Precision:            0.837848
Recall:               0.814612
F1:                   0.826067

Actual positive rate:   0.186420
Predicted positive rate: 0.181250

Confusion matrix:
[[78419  2939]
 [ 3456 15186]]

----------------------------------------------------------------------
NOISE RATE p = 0.020
------

In [23]:
# ================================================================
# CELL 17 — COMPLETE DECODER COMPARISON
# Fixed code distance: d=3
# ================================================================

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

print("=" * 90)
print("D=3 DECODER COMPARISON")
print("=" * 90)

# ---------------------------------------------------------------
# Make sure the MLP is in evaluation mode
# ---------------------------------------------------------------
model.eval()

comparison_rows = []

# Store MLP predictions so we can reuse them later
mlp_test_results = {}

# ---------------------------------------------------------------
# Evaluate each noise rate
# ---------------------------------------------------------------
for p in NOISE_RATES:

    print(f"\nProcessing p = {p:.3f}")

    # ============================================================
    # Ground-truth test labels
    # ============================================================
    y_true = test_data[p]["y"]

    # ============================================================
    # 1. Majority baseline
    # ============================================================
    majority_pred = np.zeros_like(y_true)

    comparison_rows.append({
        "distance": DISTANCE,
        "noise_rate": p,
        "decoder": "Majority baseline",
        "accuracy": accuracy_score(y_true, majority_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, majority_pred
        ),
        "precision": precision_score(
            y_true, majority_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, majority_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, majority_pred, zero_division=0
        ),
    })

    # ============================================================
    # 2. MLP v2
    # ============================================================
    X_test = test_data[p]["X"]

    X_test_tensor = torch.tensor(
        X_test,
        dtype=torch.float32
    )

    test_loader = DataLoader(
        TensorDataset(X_test_tensor),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    mlp_predictions = []

    with torch.no_grad():

        for (batch_X,) in test_loader:

            batch_X = batch_X.to(DEVICE)

            logits = model(batch_X)

            probabilities = torch.sigmoid(logits)

            predictions = (
                probabilities >= 0.5
            ).long()

            mlp_predictions.append(
                predictions.cpu().numpy()
            )

    mlp_pred = np.concatenate(
        mlp_predictions
    ).astype(int)

    # Save predictions for later error analysis
    mlp_test_results[p] = {
        "predictions": mlp_pred
    }

    comparison_rows.append({
        "distance": DISTANCE,
        "noise_rate": p,
        "decoder": "MLP v2",
        "accuracy": accuracy_score(
            y_true, mlp_pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, mlp_pred
        ),
        "precision": precision_score(
            y_true, mlp_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, mlp_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, mlp_pred, zero_division=0
        ),
    })

    # ============================================================
    # 3. PyMatching
    # ============================================================
    pymatching_pred = py_matching_results[p]["predictions"]

    comparison_rows.append({
        "distance": DISTANCE,
        "noise_rate": p,
        "decoder": "PyMatching",
        "accuracy": accuracy_score(
            y_true, pymatching_pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, pymatching_pred
        ),
        "precision": precision_score(
            y_true, pymatching_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, pymatching_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, pymatching_pred, zero_division=0
        ),
    })

# ---------------------------------------------------------------
# Create comparison DataFrame
# ---------------------------------------------------------------
comparison_df = pd.DataFrame(
    comparison_rows
)

# ---------------------------------------------------------------
# Display percentages
# ---------------------------------------------------------------
comparison_display = comparison_df.copy()

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
]

for column in metric_columns:
    comparison_display[column] = (
        comparison_display[column] * 100
    ).round(3)

print("\n")
print("=" * 90)
print("RESULTS — ALL METRICS IN PERCENT")
print("=" * 90)

display(
    comparison_display[
        [
            "distance",
            "noise_rate",
            "decoder",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
        ]
    ]
)

D=3 DECODER COMPARISON

Processing p = 0.005

Processing p = 0.010

Processing p = 0.020


RESULTS — ALL METRICS IN PERCENT


,distance,noise_rate,decoder,accuracy,balanced_accuracy,precision,recall,f1
0,3,0.005,Majority baseline,89.480,50.000,0.000,0.000,0.000
1,3,0.005,MLP v2,96.581,95.795,77.641,94.800,85.367
2,3,0.005,PyMatching,98.113,94.613,91.742,90.181,90.954
3,3,0.010,Majority baseline,81.358,50.000,0.000,0.000,0.000
4,3,0.010,MLP v2,89.126,89.736,64.909,90.709,75.670
5,3,0.010,PyMatching,93.605,88.924,83.785,81.461,82.607
6,3,0.020,Majority baseline,69.297,50.000,0.000,0.000,0.000
7,3,0.020,MLP v2,73.058,76.811,53.808,86.536,66.356
8,3,0.020,PyMatching,81.066,77.203,69.954,67.192,68.545


In [24]:
comparison_path = os.path.join(
    RESULTS_DIR,
    "d3_decoder_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print(f"Saved comparison results to:")
print(comparison_path)

Saved comparison results to:
/home/ankit/Desktop/qskit/qec_ml/results/d3_decoder_comparison.csv


In [25]:
print("=" * 90)
print("BEST DECODER BY METRIC — d=3")
print("=" * 90)

for p in NOISE_RATES:
    subset = comparison_df[
        comparison_df["noise_rate"] == p
    ].copy()

    print(f"\nNoise rate p = {p:.3f}")

    for metric in [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
    ]:
        best_idx = subset[metric].idxmax()
        best_row = subset.loc[best_idx]

        print(
            f"{metric:20s}: "
            f"{best_row['decoder']:18s} "
            f"({best_row[metric]:.4f})"
        )

BEST DECODER BY METRIC — d=3

Noise rate p = 0.005
accuracy            : PyMatching         (0.9811)
balanced_accuracy   : MLP v2             (0.9580)
precision           : PyMatching         (0.9174)
recall              : MLP v2             (0.9480)
f1                  : PyMatching         (0.9095)

Noise rate p = 0.010
accuracy            : PyMatching         (0.9361)
balanced_accuracy   : MLP v2             (0.8974)
precision           : PyMatching         (0.8378)
recall              : MLP v2             (0.9071)
f1                  : PyMatching         (0.8261)

Noise rate p = 0.020
accuracy            : PyMatching         (0.8107)
balanced_accuracy   : PyMatching         (0.7720)
precision           : PyMatching         (0.6995)
recall              : MLP v2             (0.8654)
f1                  : PyMatching         (0.6855)


In [ ]:
# The pattern is very clear:

# MLP

# aggressive
# high recall
# many false positives

# PyMatching

# more conservative
# higher precision
# substantially fewer false positives

# This is especially obvious at \(p=0.020\).

In [26]:
# ================================================================
# CELL 20 — D=3 DECODER ERROR ANALYSIS
# ================================================================

error_analysis_rows = []

for p in NOISE_RATES:

    y_true = test_data[p]["y"]

    # ------------------------------------------------------------
    # MLP predictions
    # ------------------------------------------------------------
    mlp_pred = mlp_test_results[p]["predictions"]

    mlp_cm = confusion_matrix(
        y_true,
        mlp_pred,
        labels=[0, 1]
    )

    mlp_tn, mlp_fp, mlp_fn, mlp_tp = mlp_cm.ravel()

    # ------------------------------------------------------------
    # PyMatching predictions
    # ------------------------------------------------------------
    pm_pred = py_matching_results[p]["predictions"]

    pm_cm = confusion_matrix(
        y_true,
        pm_pred,
        labels=[0, 1]
    )

    pm_tn, pm_fp, pm_fn, pm_tp = pm_cm.ravel()

    # ------------------------------------------------------------
    # MLP rates
    # ------------------------------------------------------------
    mlp_fpr = mlp_fp / (mlp_fp + mlp_tn)
    mlp_fnr = mlp_fn / (mlp_fn + mlp_tp)

    # ------------------------------------------------------------
    # PyMatching rates
    # ------------------------------------------------------------
    pm_fpr = pm_fp / (pm_fp + pm_tn)
    pm_fnr = pm_fn / (pm_fn + pm_tp)

    error_analysis_rows.append({
        "distance": DISTANCE,
        "noise_rate": p,

        "MLP_predicted_positive_rate":
            mlp_pred.mean(),

        "MLP_false_positive_rate":
            mlp_fpr,

        "MLP_false_negative_rate":
            mlp_fnr,

        "PyMatching_predicted_positive_rate":
            pm_pred.mean(),

        "PyMatching_false_positive_rate":
            pm_fpr,

        "PyMatching_false_negative_rate":
            pm_fnr,
    })


error_analysis_df = pd.DataFrame(
    error_analysis_rows
)

# ---------------------------------------------------------------
# Display percentages
# ---------------------------------------------------------------
display_df = error_analysis_df.copy()

rate_columns = [
    "MLP_predicted_positive_rate",
    "MLP_false_positive_rate",
    "MLP_false_negative_rate",
    "PyMatching_predicted_positive_rate",
    "PyMatching_false_positive_rate",
    "PyMatching_false_negative_rate",
]

for column in rate_columns:
    display_df[column] = (
        display_df[column] * 100
    ).round(3)

print("=" * 120)
print("D=3 DECODER ERROR ANALYSIS — ALL RATES IN PERCENT")
print("=" * 120)

display(display_df)

D=3 DECODER ERROR ANALYSIS — ALL RATES IN PERCENT


,distance,noise_rate,MLP_predicted_positive_rate,MLP_false_positive_rate,MLP_false_negative_rate,PyMatching_predicted_positive_rate,PyMatching_false_positive_rate,PyMatching_false_negative_rate
0,3,0.005,12.845,3.210,5.200,10.341,0.954,9.819
1,3,0.010,26.052,11.237,9.291,18.125,3.612,18.539
2,3,0.020,49.377,32.913,13.464,29.491,12.787,32.808


In [27]:
# ================================================================
# CELL 21 — MLP VALIDATION PROBABILITY ANALYSIS
# Fixed code distance: d=3
# Test sets remain completely untouched
# ================================================================

model.eval()

validation_probability_results = {}

print("=" * 90)
print("MLP VALIDATION PROBABILITY ANALYSIS — d=3")
print("=" * 90)

with torch.no_grad():

    val_logits_list = []

    for batch_X, _ in val_loader:

        batch_X = batch_X.to(DEVICE)

        logits = model(batch_X)

        val_logits_list.append(
            logits.cpu().numpy()
        )

    val_logits = np.concatenate(
        val_logits_list
    )

val_probabilities = 1.0 / (
    1.0 + np.exp(-val_logits)
)

# Store for later threshold analysis
validation_probability_results["probabilities"] = (
    val_probabilities
)

validation_probability_results["labels"] = y_val.copy()

print(f"\nValidation samples: {len(y_val):,}")
print(f"Probability range:  "
      f"{val_probabilities.min():.6f} → "
      f"{val_probabilities.max():.6f}")

print("\nProbability statistics:")
print(
    f"Mean probability:    {val_probabilities.mean():.6f}"
)

print(
    f"Median probability:  {np.median(val_probabilities):.6f}"
)

# ---------------------------------------------------------------
# Probability distributions by true class
# ---------------------------------------------------------------

for label in [0, 1]:

    class_probs = val_probabilities[
        y_val == label
    ]

    print("\n" + "-" * 70)

    print(
        f"True class = {label}"
    )

    print(
        f"Samples:             {len(class_probs):,}"
    )

    print(
        f"Mean probability:    {class_probs.mean():.6f}"
    )

    print(
        f"Median probability:  {np.median(class_probs):.6f}"
    )

    print(
        f"25th percentile:     "
        f"{np.percentile(class_probs, 25):.6f}"
    )

    print(
        f"75th percentile:     "
        f"{np.percentile(class_probs, 75):.6f}"
    )

    print(
        f"Minimum:             {class_probs.min():.6f}"
    )

    print(
        f"Maximum:             {class_probs.max():.6f}"
    )

print("\n" + "=" * 90)
print("ANALYSIS COMPLETE")
print("=" * 90)

MLP VALIDATION PROBABILITY ANALYSIS — d=3

Validation samples: 15,000
Probability range:  0.000224 → 0.998037

Probability statistics:
Mean probability:    0.299926
Median probability:  0.088994

----------------------------------------------------------------------
True class = 0
Samples:             12,028
Mean probability:    0.172143
Median probability:  0.021867
25th percentile:     0.000224
75th percentile:     0.267259
Minimum:             0.000224
Maximum:             0.996262

----------------------------------------------------------------------
True class = 1
Samples:             2,972
Mean probability:    0.817080
Median probability:  0.897773
25th percentile:     0.729071
75th percentile:     0.984164
Minimum:             0.000224
Maximum:             0.998037

ANALYSIS COMPLETE


In [28]:
# ================================================================
# CELL 22 — VALIDATION THRESHOLD SWEEP
# ================================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

val_probs = validation_probability_results["probabilities"]
val_labels = validation_probability_results["labels"]

thresholds = np.arange(
    0.10,
    0.91,
    0.05
)

threshold_rows = []

for threshold in thresholds:

    predictions = (
        val_probs >= threshold
    ).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            val_labels,
            predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            val_labels,
            predictions
        ),
        "precision": precision_score(
            val_labels,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            val_labels,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            val_labels,
            predictions,
            zero_division=0
        ),
        "predicted_positive_rate": predictions.mean(),
    })

threshold_df = pd.DataFrame(
    threshold_rows
)

display_df = threshold_df.copy()

for column in [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "predicted_positive_rate",
]:
    display_df[column] = (
        display_df[column] * 100
    ).round(3)

print("=" * 100)
print("VALIDATION THRESHOLD SWEEP — d=3")
print("=" * 100)

display(display_df)

VALIDATION THRESHOLD SWEEP — d=3


,threshold,accuracy,balanced_accuracy,precision,recall,f1,predicted_positive_rate
0,0.10,70.613,81.157,40.162,98.620,57.079,48.653
1,0.15,74.267,83.295,43.401,98.250,60.206,44.853
2,0.20,76.873,84.629,46.050,97.476,62.550,41.940
3,0.25,78.680,85.414,48.106,96.568,64.220,39.773
4,0.30,80.233,86.015,50.062,95.592,65.711,37.833
5,0.35,82.093,86.706,52.687,94.347,67.615,35.480
6,0.40,83.893,87.398,55.578,93.203,69.633,33.227
7,0.45,84.987,87.662,57.573,92.093,70.852,31.693
8,0.50,86.393,87.994,60.444,90.646,72.527,29.713
9,0.55,87.373,87.630,62.969,88.055,73.429,27.707


In [29]:
# ================================================================
# CELL 23 — FINE THRESHOLD CHECK AROUND 0.5
# Validation set only
# Fixed distance: d=3
# ================================================================

fine_thresholds = np.arange(
    0.40,
    0.601,
    0.01
)

fine_threshold_rows = []

for threshold in fine_thresholds:

    predictions = (
        val_probs >= threshold
    ).astype(int)

    fine_threshold_rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            val_labels,
            predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            val_labels,
            predictions
        ),
        "precision": precision_score(
            val_labels,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            val_labels,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            val_labels,
            predictions,
            zero_division=0
        ),
        "predicted_positive_rate": predictions.mean(),
    })

fine_threshold_df = pd.DataFrame(
    fine_threshold_rows
)

# Find best threshold according to our primary metric
best_idx = fine_threshold_df[
    "balanced_accuracy"
].idxmax()

best_threshold_row = fine_threshold_df.loc[
    best_idx
]

print("=" * 90)
print("FINE THRESHOLD CHECK — d=3")
print("=" * 90)

print(
    f"\nBest validation threshold: "
    f"{best_threshold_row['threshold']:.2f}"
)

print(
    f"Balanced accuracy: "
    f"{best_threshold_row['balanced_accuracy']:.6f}"
)

print(
    f"Accuracy: "
    f"{best_threshold_row['accuracy']:.6f}"
)

print(
    f"Precision: "
    f"{best_threshold_row['precision']:.6f}"
)

print(
    f"Recall: "
    f"{best_threshold_row['recall']:.6f}"
)

print(
    f"F1: "
    f"{best_threshold_row['f1']:.6f}"
)

print(
    f"Predicted positive rate: "
    f"{best_threshold_row['predicted_positive_rate']:.6f}"
)

print("\nThresholds near the optimum:\n")

near_optimum = fine_threshold_df[
    (fine_threshold_df["threshold"] >= 0.45) &
    (fine_threshold_df["threshold"] <= 0.55)
].copy()

display(
    near_optimum.round(4)
)

print("=" * 90)

FINE THRESHOLD CHECK — d=3

Best validation threshold: 0.50
Balanced accuracy: 0.879943
Accuracy: 0.863933
Precision: 0.604442
Recall: 0.906460
F1: 0.725266
Predicted positive rate: 0.297133

Thresholds near the optimum:



,threshold,accuracy,balanced_accuracy,precision,recall,f1,predicted_positive_rate
5,0.45,0.8499,0.8766,0.5757,0.9209,0.7085,0.3169
6,0.46,0.8527,0.8778,0.5812,0.9192,0.7121,0.3134
7,0.47,0.8553,0.8782,0.5862,0.9162,0.7150,0.3097
8,0.48,0.8578,0.8789,0.5913,0.9139,0.7180,0.3062
9,0.49,0.8611,0.8793,0.5984,0.9095,0.7219,0.3011
10,0.50,0.8639,0.8799,0.6044,0.9065,0.7253,0.2971
11,0.51,0.8659,0.8798,0.6089,0.9028,0.7273,0.2937
12,0.52,0.8679,0.8796,0.6137,0.8991,0.7295,0.2903
13,0.53,0.8697,0.8786,0.6184,0.8933,0.7309,0.2862
14,0.54,0.8716,0.8774,0.6238,0.8869,0.7324,0.2817


In [30]:
# ================================================================
# CELL 24 — MLP vs PYMATCHING DISAGREEMENT ANALYSIS
# Fixed code distance: d=3
# ================================================================

print("=" * 90)
print("MLP vs PYMATCHING DISAGREEMENT ANALYSIS — d=3")
print("=" * 90)

disagreement_rows = []

for p in NOISE_RATES:

    y_true = test_data[p]["y"]

    mlp_pred = mlp_test_results[p]["predictions"]
    pm_pred = py_matching_results[p]["predictions"]

    # ------------------------------------------------------------
    # Correctness masks
    # ------------------------------------------------------------
    mlp_correct = (mlp_pred == y_true)
    pm_correct = (pm_pred == y_true)

    both_correct = mlp_correct & pm_correct
    mlp_only_correct = mlp_correct & ~pm_correct
    pm_only_correct = ~mlp_correct & pm_correct
    both_wrong = ~mlp_correct & ~pm_correct

    # ------------------------------------------------------------
    # Agreement
    # ------------------------------------------------------------
    agreement = (mlp_pred == pm_pred)

    # ------------------------------------------------------------
    # Counts
    # ------------------------------------------------------------
    total = len(y_true)

    counts = {
        "both_correct": np.sum(both_correct),
        "mlp_only_correct": np.sum(mlp_only_correct),
        "pymatching_only_correct": np.sum(pm_only_correct),
        "both_wrong": np.sum(both_wrong),
    }

    print("\n" + "-" * 90)
    print(f"NOISE RATE p = {p:.3f}")
    print("-" * 90)

    print(f"Total test shots:             {total:,}")

    print(
        f"Decoder agreement:            "
        f"{agreement.mean() * 100:.3f}%"
    )

    print(
        f"Decoder disagreement:         "
        f"{(~agreement).mean() * 100:.3f}%"
    )

    print()

    print(
        f"Both correct:                 "
        f"{counts['both_correct']:,} "
        f"({counts['both_correct'] / total * 100:.3f}%)"
    )

    print(
        f"MLP correct / PyMatching wrong:"
        f" {counts['mlp_only_correct']:,} "
        f"({counts['mlp_only_correct'] / total * 100:.3f}%)"
    )

    print(
        f"PyMatching correct / MLP wrong:"
        f" {counts['pymatching_only_correct']:,} "
        f"({counts['pymatching_only_correct'] / total * 100:.3f}%)"
    )

    print(
        f"Both wrong:                   "
        f"{counts['both_wrong']:,} "
        f"({counts['both_wrong'] / total * 100:.3f}%)"
    )

    disagreement_rows.append({
        "distance": DISTANCE,
        "noise_rate": p,
        "agreement_rate": agreement.mean(),
        "disagreement_rate": (~agreement).mean(),
        "both_correct_rate": both_correct.mean(),
        "mlp_only_correct_rate": mlp_only_correct.mean(),
        "pymatching_only_correct_rate": pm_only_correct.mean(),
        "both_wrong_rate": both_wrong.mean(),
    })

disagreement_df = pd.DataFrame(
    disagreement_rows
)

print("\n" + "=" * 90)
print("DISAGREEMENT ANALYSIS COMPLETE")
print("=" * 90)

MLP vs PYMATCHING DISAGREEMENT ANALYSIS — d=3

------------------------------------------------------------------------------------------
NOISE RATE p = 0.005
------------------------------------------------------------------------------------------
Total test shots:             100,000
Decoder agreement:            97.354%
Decoder disagreement:         2.646%

Both correct:                 96,024 (96.024%)
MLP correct / PyMatching wrong: 557 (0.557%)
PyMatching correct / MLP wrong: 2,089 (2.089%)
Both wrong:                   1,330 (1.330%)

------------------------------------------------------------------------------------------
NOISE RATE p = 0.010
------------------------------------------------------------------------------------------
Total test shots:             100,000
Decoder agreement:            91.361%
Decoder disagreement:         8.639%

Both correct:                 87,046 (87.046%)
MLP correct / PyMatching wrong: 2,080 (2.080%)
PyMatching correct / MLP wrong: 6,559 (6

In [31]:
# ================================================================
# CELL 25 — DISAGREEMENT ANALYSIS BY TRUE LOGICAL CLASS
# Fixed code distance: d=3
# ================================================================

print("=" * 100)
print("D=3 DECODER DISAGREEMENT BY TRUE LOGICAL CLASS")
print("=" * 100)

class_disagreement_rows = []

for p in NOISE_RATES:

    y_true = test_data[p]["y"]

    mlp_pred = mlp_test_results[p]["predictions"]
    pm_pred = py_matching_results[p]["predictions"]

    print("\n" + "-" * 100)
    print(f"NOISE RATE p = {p:.3f}")
    print("-" * 100)

    for true_class in [0, 1]:

        mask = (y_true == true_class)

        y_class = y_true[mask]
        mlp_class = mlp_pred[mask]
        pm_class = pm_pred[mask]

        mlp_correct = (mlp_class == y_class)
        pm_correct = (pm_class == y_class)

        both_correct = mlp_correct & pm_correct
        mlp_only_correct = mlp_correct & ~pm_correct
        pm_only_correct = ~mlp_correct & pm_correct
        both_wrong = ~mlp_correct & ~pm_correct

        n = len(y_class)

        print(f"\nTrue logical class = {true_class}")
        print(f"Samples: {n:,}")

        print(
            f"  Both correct:                  "
            f"{both_correct.sum():,} "
            f"({both_correct.mean() * 100:.3f}%)"
        )

        print(
            f"  MLP correct / PyMatching wrong: "
            f"{mlp_only_correct.sum():,} "
            f"({mlp_only_correct.mean() * 100:.3f}%)"
        )

        print(
            f"  PyMatching correct / MLP wrong: "
            f"{pm_only_correct.sum():,} "
            f"({pm_only_correct.mean() * 100:.3f}%)"
        )

        print(
            f"  Both wrong:                    "
            f"{both_wrong.sum():,} "
            f"({both_wrong.mean() * 100:.3f}%)"
        )

        class_disagreement_rows.append({
            "distance": DISTANCE,
            "noise_rate": p,
            "true_class": true_class,
            "samples": n,
            "both_correct_rate": both_correct.mean(),
            "mlp_only_correct_rate": mlp_only_correct.mean(),
            "pymatching_only_correct_rate": pm_only_correct.mean(),
            "both_wrong_rate": both_wrong.mean(),
        })

class_disagreement_df = pd.DataFrame(
    class_disagreement_rows
)

print("\n" + "=" * 100)
print("CLASS-CONDITIONAL DISAGREEMENT ANALYSIS COMPLETE")
print("=" * 100)

D=3 DECODER DISAGREEMENT BY TRUE LOGICAL CLASS

----------------------------------------------------------------------------------------------------
NOISE RATE p = 0.005
----------------------------------------------------------------------------------------------------

True logical class = 0
Samples: 89,480
  Both correct:                  86,562 (96.739%)
  MLP correct / PyMatching wrong: 46 (0.051%)
  PyMatching correct / MLP wrong: 2,064 (2.307%)
  Both wrong:                    808 (0.903%)

True logical class = 1
Samples: 10,520
  Both correct:                  9,462 (89.943%)
  MLP correct / PyMatching wrong: 511 (4.857%)
  PyMatching correct / MLP wrong: 25 (0.238%)
  Both wrong:                    522 (4.962%)

----------------------------------------------------------------------------------------------------
NOISE RATE p = 0.010
----------------------------------------------------------------------------------------------------

True logical class = 0
Samples: 81,358
  Both

In [33]:
# ================================================================
# CELL 26 — SYNDROME-WEIGHT ANALYSIS OF DECODER ERRORS
# Fixed code distance: d=3
# ================================================================

import pandas as pd
import numpy as np

print("=" * 100)
print("D=3 SYNDROME-WEIGHT ANALYSIS OF DECODER DISAGREEMENT")
print("=" * 100)

syndrome_weight_rows = []

for p in NOISE_RATES:

    X = test_data[p]["X"]
    y_true = test_data[p]["y"]

    mlp_pred = mlp_test_results[p]["predictions"]
    pm_pred = py_matching_results[p]["predictions"]

    # Number of triggered detectors in each shot
    syndrome_weight = X.sum(axis=1)

    print("\n" + "-" * 100)
    print(f"NOISE RATE p = {p:.3f}")
    print("-" * 100)

    for weight in np.unique(syndrome_weight):

        mask = syndrome_weight == weight
        n = mask.sum()

        y_w = y_true[mask]
        mlp_w = mlp_pred[mask]
        pm_w = pm_pred[mask]

        mlp_correct = mlp_w == y_w
        pm_correct = pm_w == y_w

        both_correct = mlp_correct & pm_correct
        mlp_only_correct = mlp_correct & ~pm_correct
        pm_only_correct = ~mlp_correct & pm_correct
        both_wrong = ~mlp_correct & ~pm_correct

        disagreement = mlp_w != pm_w

        syndrome_weight_rows.append({
            "distance": DISTANCE,
            "noise_rate": p,
            "syndrome_weight": int(weight),
            "samples": int(n),
            "fraction_of_test": n / len(y_true),
            "logical_positive_rate": y_w.mean(),
            "mlp_accuracy": mlp_correct.mean(),
            "pymatching_accuracy": pm_correct.mean(),
            "agreement_rate": (~disagreement).mean(),
            "disagreement_rate": disagreement.mean(),
            "both_correct_rate": both_correct.mean(),
            "mlp_only_correct_rate": mlp_only_correct.mean(),
            "pymatching_only_correct_rate": pm_only_correct.mean(),
            "both_wrong_rate": both_wrong.mean(),
        })

        print(
            f"weight={int(weight):2d} | "
            f"N={n:6,d} | "
            f"logical-1={y_w.mean()*100:6.2f}% | "
            f"MLP acc={mlp_correct.mean()*100:6.2f}% | "
            f"PM acc={pm_correct.mean()*100:6.2f}% | "
            f"disagreement={disagreement.mean()*100:6.2f}%"
        )

syndrome_weight_df = pd.DataFrame(syndrome_weight_rows)

print("\n" + "=" * 100)
print("SYNDROME-WEIGHT ANALYSIS COMPLETE")
print("=" * 100)

D=3 SYNDROME-WEIGHT ANALYSIS OF DECODER DISAGREEMENT

----------------------------------------------------------------------------------------------------
NOISE RATE p = 0.005
----------------------------------------------------------------------------------------------------
weight= 0 | N=42,192 | logical-1=  0.00% | MLP acc=100.00% | PM acc=100.00% | disagreement=  0.00%
weight= 1 | N=13,721 | logical-1= 25.70% | MLP acc= 99.42% | PM acc= 99.42% | disagreement=  0.00%
weight= 2 | N=21,791 | logical-1=  9.45% | MLP acc= 98.36% | PM acc= 98.77% | disagreement=  0.46%
weight= 3 | N=10,120 | logical-1= 23.47% | MLP acc= 92.11% | PM acc= 95.76% | disagreement=  5.60%
weight= 4 | N= 7,011 | logical-1= 16.94% | MLP acc= 88.35% | PM acc= 93.58% | disagreement=  9.34%
weight= 5 | N= 2,896 | logical-1= 27.42% | MLP acc= 77.94% | PM acc= 89.47% | disagreement= 20.51%
weight= 6 | N= 1,452 | logical-1= 22.73% | MLP acc= 72.87% | PM acc= 88.22% | disagreement= 27.20%
weight= 7 | N=   529 | logical

In [35]:
# One particularly important observation

# Look at the logical-1 rate as syndrome weight increases.

# At \(p=0.020\):

# weight 1 → 27.06%
# weight 3 → 30.45%
# weight 5 → 34.26%
# weight 7 → 37.83%
# weight 9 → 39.82%
# weight 11 → 43.96%
# weight 13 → 48.12%
# weight 15 → 62.50%

# There is a general upward trend.

# That means syndrome weight isn't merely correlated with decoder difficulty; it also contains information about the logical-error class.

# We should not conclude that high syndrome weight itself causes the MLP to fail.

# Syndrome weight is only a coarse statistic: $$ w(s)=\sum_i s_i. $$

# Two syndromes can have exactly the same weight but completely different detector locations:

# $$ s_1=(1,1,0,0,\ldots) $$ and $$ s_2=(0,0,1,1,\ldots). $$

# Their weights are both 2, but their spatial/temporal structure can be completely different.

In [ ]:
# The key question is:

# For syndromes with the same weight, do the MLP and PyMatching disagree on particular detector patterns?

# We can start with a simple and very useful analysis:

# Group samples by syndrome weight.
# For each weight, examine MLP-only correct and PyMatching-only correct cases.
# Calculate the activation frequency of each of the 24 detectors.
# Compare the detector-frequency profiles.

# This can reveal whether the MLP's successes/failures are associated with particular detector locations.

# Use test data descriptively only here—we are still doing error analysis, not selecting a model.

In [36]:
# ================================================================
# CELL 27 — DETECTOR ACTIVATION ANALYSIS FOR DECODER-SPECIFIC CASES
# Fixed code distance: d=3
# ================================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("D=3 DETECTOR ACTIVATION ANALYSIS")
print("=" * 100)

detector_analysis_rows = []

for p in NOISE_RATES:

    X = test_data[p]["X"]
    y_true = test_data[p]["y"]

    mlp_pred = mlp_test_results[p]["predictions"]
    pm_pred = py_matching_results[p]["predictions"]

    mlp_correct = mlp_pred == y_true
    pm_correct = pm_pred == y_true

    # Four mutually exclusive cases
    cases = {
        "both_correct": mlp_correct & pm_correct,
        "mlp_only_correct": mlp_correct & ~pm_correct,
        "pymatching_only_correct": ~mlp_correct & pm_correct,
        "both_wrong": ~mlp_correct & ~pm_correct,
    }

    print("\n" + "-" * 100)
    print(f"NOISE RATE p = {p:.3f}")
    print("-" * 100)

    for case_name, mask in cases.items():

        X_case = X[mask]

        if len(X_case) == 0:
            print(f"\n{case_name}: no samples")
            continue

        detector_rates = X_case.mean(axis=0)

        print(f"\n{case_name}")
        print(f"Samples: {len(X_case):,}")

        print(
            f"Mean syndrome weight: "
            f"{X_case.sum(axis=1).mean():.3f}"
        )

        print("Detector activation rates:")

        for detector_id, rate in enumerate(detector_rates):
            print(
                f"  Detector {detector_id:2d}: "
                f"{rate * 100:6.2f}%"
            )

        for detector_id, rate in enumerate(detector_rates):

            detector_analysis_rows.append({
                "distance": DISTANCE,
                "noise_rate": p,
                "case": case_name,
                "detector": detector_id,
                "activation_rate": rate,
                "samples": len(X_case),
            })


detector_analysis_df = pd.DataFrame(
    detector_analysis_rows
)

print("\n" + "=" * 100)
print("DETECTOR ACTIVATION ANALYSIS COMPLETE")
print("=" * 100)

D=3 DETECTOR ACTIVATION ANALYSIS

----------------------------------------------------------------------------------------------------
NOISE RATE p = 0.005
----------------------------------------------------------------------------------------------------

both_correct
Samples: 96,024
Mean syndrome weight: 1.330
Detector activation rates:
  Detector  0:   3.53%
  Detector  1:   6.51%
  Detector  2:   5.76%
  Detector  3:   4.19%
  Detector  4:   5.44%
  Detector  5:   6.77%
  Detector  6:   7.63%
  Detector  7:   4.26%
  Detector  8:   4.33%
  Detector  9:   7.52%
  Detector 10:   6.86%
  Detector 11:   5.32%
  Detector 12:   5.28%
  Detector 13:   6.80%
  Detector 14:   7.69%
  Detector 15:   4.30%
  Detector 16:   4.29%
  Detector 17:   7.49%
  Detector 18:   6.90%
  Detector 19:   5.41%
  Detector 20:   3.66%
  Detector 21:   4.69%
  Detector 22:   5.42%
  Detector 23:   2.98%

mlp_only_correct
Samples: 557
Mean syndrome weight: 4.908
Detector activation rates:
  Detector  0:   7.1

In [37]:
# Decoder disagreement is concentrated in more complex syndromes.
# Hypothesis:
    # The MLP-only and PyMatching-only cases occupy broadly similar high-weight regions of syndrome space, but the two decoders make different decisions within those regions.

# Suppose MLP-only cases have mean weight 6.78 and PM-only cases have mean weight 6.49.

# That difference could simply explain some detector-frequency differences.

# We therefore want to compare the decoders at the same syndrome weight.

# For example: $$ w(s)=6 $$

#         and then compare:
#         MLP-only correct
#         PyMatching-only correct
#         both correct both wrong 

#     within weight 6 only.

# This removes the biggest confounding factor.

In [38]:
# ================================================================
# CELL 28 — WEIGHT-CONTROLLED DETECTOR COMPARISON
# Fixed code distance: d=3
# ================================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("D=3 WEIGHT-CONTROLLED DETECTOR COMPARISON")
print("=" * 100)

weight_controlled_rows = []

for p in NOISE_RATES:

    X = test_data[p]["X"]
    y_true = test_data[p]["y"]

    mlp_pred = mlp_test_results[p]["predictions"]
    pm_pred = py_matching_results[p]["predictions"]

    mlp_correct = mlp_pred == y_true
    pm_correct = pm_pred == y_true

    mlp_only = mlp_correct & ~pm_correct
    pm_only = ~mlp_correct & pm_correct

    syndrome_weight = X.sum(axis=1)

    print("\n" + "-" * 100)
    print(f"NOISE RATE p = {p:.3f}")
    print("-" * 100)

    for weight in sorted(np.unique(syndrome_weight)):

        mask_mlp = mlp_only & (syndrome_weight == weight)
        mask_pm = pm_only & (syndrome_weight == weight)

        n_mlp = int(mask_mlp.sum())
        n_pm = int(mask_pm.sum())

        # Ignore very small groups
        if n_mlp < 20 or n_pm < 20:
            continue

        mlp_rates = X[mask_mlp].mean(axis=0)
        pm_rates = X[mask_pm].mean(axis=0)

        difference = mlp_rates - pm_rates

        for detector_id in range(X.shape[1]):

            weight_controlled_rows.append({
                "distance": DISTANCE,
                "noise_rate": p,
                "syndrome_weight": int(weight),
                "detector": detector_id,
                "mlp_only_samples": n_mlp,
                "pymatching_only_samples": n_pm,
                "mlp_only_activation": mlp_rates[detector_id],
                "pymatching_only_activation": pm_rates[detector_id],
                "activation_difference": difference[detector_id],
                "absolute_difference": abs(difference[detector_id]),
            })

        # Show the strongest detector differences
        ranking = np.argsort(np.abs(difference))[::-1]

        print(
            f"\nSyndrome weight = {int(weight)} | "
            f"MLP-only N={n_mlp:,} | "
            f"PM-only N={n_pm:,}"
        )

        print("Top detector activation differences:")

        for detector_id in ranking[:5]:

            print(
                f"  Detector {detector_id:2d}: "
                f"MLP={mlp_rates[detector_id] * 100:6.2f}% | "
                f"PM={pm_rates[detector_id] * 100:6.2f}% | "
                f"Δ={difference[detector_id] * 100:+6.2f}%"
            )


weight_controlled_df = pd.DataFrame(
    weight_controlled_rows
)

print("\n" + "=" * 100)
print("WEIGHT-CONTROLLED DETECTOR ANALYSIS COMPLETE")
print("=" * 100)

print(
    f"\nGenerated rows: {len(weight_controlled_df):,}"
)

D=3 WEIGHT-CONTROLLED DETECTOR COMPARISON

----------------------------------------------------------------------------------------------------
NOISE RATE p = 0.005
----------------------------------------------------------------------------------------------------

Syndrome weight = 3 | MLP-only N=99 | PM-only N=468
Top detector activation differences:
  Detector  6: MLP= 43.43% | PM= 16.24% | Δ=+27.20%
  Detector  9: MLP= 36.36% | PM= 22.86% | Δ=+13.50%
  Detector  2: MLP= 12.12% | PM= 22.44% | Δ=-10.31%
  Detector 15: MLP=  3.03% | PM= 10.47% | Δ= -7.44%
  Detector 13: MLP=  3.03% | PM=  9.40% | Δ= -6.37%

Syndrome weight = 4 | MLP-only N=144 | PM-only N=511
Top detector activation differences:
  Detector  8: MLP= 19.44% | PM=  7.44% | Δ=+12.01%
  Detector 20: MLP=  4.86% | PM= 16.63% | Δ=-11.77%
  Detector  2: MLP= 13.19% | PM= 23.09% | Δ= -9.90%
  Detector 16: MLP= 20.83% | PM= 11.15% | Δ= +9.68%
  Detector 10: MLP= 22.92% | PM= 13.89% | Δ= +9.02%

Syndrome weight = 5 | MLP-only N

In [ ]:
    # Raw syndrome
    #     ↓
    # Class imbalance
    #     ↓
    # MLP baseline
    #     ↓
    # PyMatching baseline
    #     ↓
    # Threshold validation
    #     ↓
    # Decoder disagreement
    #     ↓
    # Class-conditional disagreement
    #     ↓
    # Syndrome-weight analysis
    #     ↓
    # Detector activation
    #     ↓
    # Weight-controlled detector structure

In [39]:
# ================================================================
# CELL 29 — CLASS + WEIGHT CONTROLLED DETECTOR COMPARISON
# Fixed code distance: d=3
# ================================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("D=3 CLASS + WEIGHT CONTROLLED DETECTOR COMPARISON")
print("=" * 100)

class_weight_rows = []

for p in NOISE_RATES:

    X = test_data[p]["X"]
    y_true = test_data[p]["y"]

    mlp_pred = mlp_test_results[p]["predictions"]
    pm_pred = py_matching_results[p]["predictions"]

    mlp_correct = mlp_pred == y_true
    pm_correct = pm_pred == y_true

    mlp_only = mlp_correct & ~pm_correct
    pm_only = ~mlp_correct & pm_correct

    syndrome_weight = X.sum(axis=1)

    print("\n" + "-" * 100)
    print(f"NOISE RATE p = {p:.3f}")
    print("-" * 100)

    for true_class in [0, 1]:

        print(f"\nTRUE LOGICAL CLASS = {true_class}")

        for weight in sorted(np.unique(syndrome_weight)):

            base_mask = (
                (y_true == true_class)
                & (syndrome_weight == weight)
            )

            mask_mlp = base_mask & mlp_only
            mask_pm = base_mask & pm_only

            n_mlp = int(mask_mlp.sum())
            n_pm = int(mask_pm.sum())

            # Require enough samples in both groups
            if n_mlp < 20 or n_pm < 20:
                continue

            mlp_rates = X[mask_mlp].mean(axis=0)
            pm_rates = X[mask_pm].mean(axis=0)

            difference = mlp_rates - pm_rates

            ranking = np.argsort(
                np.abs(difference)
            )[::-1]

            print(
                f"\nWeight = {int(weight):2d} | "
                f"MLP-only N={n_mlp:,} | "
                f"PM-only N={n_pm:,}"
            )

            print(
                "Top detector activation differences:"
            )

            for detector_id in ranking[:5]:

                print(
                    f"  Detector {detector_id:2d}: "
                    f"MLP={mlp_rates[detector_id] * 100:6.2f}% | "
                    f"PM={pm_rates[detector_id] * 100:6.2f}% | "
                    f"Δ={difference[detector_id] * 100:+6.2f}%"
                )

                class_weight_rows.append({
                    "distance": DISTANCE,
                    "noise_rate": p,
                    "true_class": true_class,
                    "syndrome_weight": int(weight),
                    "detector": detector_id,
                    "mlp_only_samples": n_mlp,
                    "pymatching_only_samples": n_pm,
                    "mlp_only_activation": mlp_rates[detector_id],
                    "pymatching_only_activation": pm_rates[detector_id],
                    "activation_difference": difference[detector_id],
                    "absolute_difference": abs(
                        difference[detector_id]
                    ),
                })


class_weight_df = pd.DataFrame(
    class_weight_rows
)

print("\n" + "=" * 100)
print("CLASS + WEIGHT CONTROLLED ANALYSIS COMPLETE")
print("=" * 100)

print(
    f"\nGenerated rows: "
    f"{len(class_weight_df):,}"
)

D=3 CLASS + WEIGHT CONTROLLED DETECTOR COMPARISON

----------------------------------------------------------------------------------------------------
NOISE RATE p = 0.005
----------------------------------------------------------------------------------------------------

TRUE LOGICAL CLASS = 0

TRUE LOGICAL CLASS = 1

----------------------------------------------------------------------------------------------------
NOISE RATE p = 0.010
----------------------------------------------------------------------------------------------------

TRUE LOGICAL CLASS = 0

Weight =  4 | MLP-only N=50 | PM-only N=1,198
Top detector activation differences:
  Detector  6: MLP= 68.00% | PM= 21.12% | Δ=+46.88%
  Detector  1: MLP=  4.00% | PM= 35.98% | Δ=-31.98%
  Detector 18: MLP= 36.00% | PM= 10.93% | Δ=+25.07%
  Detector 17: MLP= 52.00% | PM= 28.55% | Δ=+23.45%
  Detector 16: MLP=  0.00% | PM= 13.69% | Δ=-13.69%

Weight =  5 | MLP-only N=60 | PM-only N=1,348
Top detector activation differences:
  

In [ ]:
# We now know:

# Decoder disagreement increases with syndrome weight.
# MLP-only and PM-only cases are concentrated in different logical classes.
# Detector activation differences remain after controlling for class and weight.
# Some detector differences are highly class-dependent.
# The exclusive-success groups can be very imbalanced.
# Therefore, detailed syndrome structure is a plausible explanation for decoder disagreement, but individual detector effects need statistical validation.


# The MLP receives the complete 24-bit syndrome: s=(s_1,…,s_24).
# "Which complete syndrome configurations lead to disagreement?"

# This is much closer to the actual decoding problem.

# For \(d=3\), there are only $$ 2^{24}=16,777,216 $$ possible binary syndromes.

# Our 100k-shot test sets won't cover all of them, but at \(p=.02\) we'll have many repeated configurations.

# We can therefore identify:
#     syndrome patterns seen multiple times,
#     whether MLP and PyMatching agree,
#     whether each prediction is correct,
#     whether the pattern is associated with logical 0 or 1.

In [40]:
# ================================================================
# CELL 30 — REPEATED SYNDROME PATTERN ANALYSIS
# Fixed code distance: d=3
# ================================================================

import numpy as np
import pandas as pd
from collections import Counter

print("=" * 100)
print("D=3 REPEATED SYNDROME PATTERN ANALYSIS")
print("=" * 100)

syndrome_pattern_rows = []

for p in NOISE_RATES:

    X = test_data[p]["X"]
    y_true = test_data[p]["y"]

    mlp_pred = mlp_test_results[p]["predictions"]
    pm_pred = py_matching_results[p]["predictions"]

    # Convert each 24-bit syndrome into a compact tuple
    syndrome_tuples = [tuple(row.astype(np.uint8)) for row in X]

    counts = Counter(syndrome_tuples)

    unique_count = len(counts)
    repeated_patterns = sum(
        count > 1 for count in counts.values()
    )

    max_frequency = max(counts.values())

    print("\n" + "-" * 100)
    print(f"NOISE RATE p = {p:.3f}")
    print("-" * 100)

    print(f"Total test shots:          {len(X):,}")
    print(f"Unique syndromes:          {unique_count:,}")
    print(f"Repeated syndromes:        {repeated_patterns:,}")
    print(f"Most frequent syndrome:    {max_frequency:,} occurrences")

    # Analyze only syndromes appearing at least 2 times
    repeated_items = [
        (syndrome, count)
        for syndrome, count in counts.items()
        if count >= 2
    ]

    # Sort by frequency
    repeated_items.sort(
        key=lambda item: item[1],
        reverse=True
    )

    print("\nTop repeated syndrome patterns:")

    for syndrome, count in repeated_items[:20]:

        mask = np.all(
            X == np.asarray(syndrome, dtype=np.uint8),
            axis=1
        )

        y_pattern = y_true[mask]
        mlp_pattern = mlp_pred[mask]
        pm_pattern = pm_pred[mask]

        mlp_accuracy = np.mean(
            mlp_pattern == y_pattern
        )

        pm_accuracy = np.mean(
            pm_pattern == y_pattern
        )

        agreement = np.mean(
            mlp_pattern == pm_pattern
        )

        syndrome_weight = int(
            np.sum(syndrome)
        )

        syndrome_pattern_rows.append({
            "distance": DISTANCE,
            "noise_rate": p,
            "syndrome": "".join(
                map(str, syndrome)
            ),
            "syndrome_weight": syndrome_weight,
            "frequency": count,
            "logical_positive_rate": y_pattern.mean(),
            "mlp_accuracy": mlp_accuracy,
            "pymatching_accuracy": pm_accuracy,
            "agreement_rate": agreement,
        })

        print(
            f"weight={syndrome_weight:2d} | "
            f"count={count:5d} | "
            f"logical-1={y_pattern.mean()*100:6.2f}% | "
            f"MLP acc={mlp_accuracy*100:6.2f}% | "
            f"PM acc={pm_accuracy*100:6.2f}% | "
            f"agreement={agreement*100:6.2f}% | "
            f"syndrome={''.join(map(str, syndrome))}"
        )


syndrome_pattern_df = pd.DataFrame(
    syndrome_pattern_rows
)

print("\n" + "=" * 100)
print("REPEATED SYNDROME PATTERN ANALYSIS COMPLETE")
print("=" * 100)

D=3 REPEATED SYNDROME PATTERN ANALYSIS

----------------------------------------------------------------------------------------------------
NOISE RATE p = 0.005
----------------------------------------------------------------------------------------------------
Total test shots:          100,000
Unique syndromes:          8,175
Repeated syndromes:        2,901
Most frequent syndrome:    42,192 occurrences

Top repeated syndrome patterns:
weight= 0 | count=42192 | logical-1=  0.00% | MLP acc=100.00% | PM acc=100.00% | agreement=100.00% | syndrome=000000000000000000000000
weight= 1 | count= 1053 | logical-1=  0.00% | MLP acc=100.00% | PM acc=100.00% | agreement=100.00% | syndrome=000000000000000000100000
weight= 1 | count= 1001 | logical-1= 99.30% | MLP acc= 99.30% | PM acc= 99.30% | agreement=100.00% | syndrome=010000000000000000000000
weight= 1 | count=  970 | logical-1=  0.00% | MLP acc=100.00% | PM acc=100.00% | agreement=100.00% | syndrome=000001000000000000000000
weight= 1 | count

In [ ]:
# The MLP is competitive with PyMatching at \(d=3\), with a different error profile: it tends toward higher recall and more false positives, whereas PyMatching is more conservative.